In [1]:
import pyspark

In [2]:
spark = (
    pyspark.sql.SparkSession.builder
        .appName("Text session")
        .master('local[4]')
        .config('spark.executor.memory', '2g')
        .config('spark.driver.memory', '4g') # only for cluster, not for local
        .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/05 15:48:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.createDataFrame([
    (0, 'Titanic disaster survival prediction'),
    (1, 'Spark ML example for text embeddings')
], ['id', 'text'])

df.show(2, truncate=False)

+---+------------------------------------+
|id |text                                |
+---+------------------------------------+
|0  |Titanic disaster survival prediction|
|1  |Spark ML example for text embeddings|
+---+------------------------------------+



## Токенизация

In [4]:
from pyspark.ml.feature import Tokenizer

In [5]:
tokenizer = Tokenizer(inputCol='text', outputCol='words')
df_tokens = tokenizer.transform(df)
df_tokens.show(3, truncate=False)

+---+------------------------------------+-------------------------------------------+
|id |text                                |words                                      |
+---+------------------------------------+-------------------------------------------+
|0  |Titanic disaster survival prediction|[titanic, disaster, survival, prediction]  |
|1  |Spark ML example for text embeddings|[spark, ml, example, for, text, embeddings]|
+---+------------------------------------+-------------------------------------------+



## ТF-IDF

In [6]:
from pyspark.ml.feature import HashingTF, IDF
# IDFModel - уже обученная модель

In [7]:
hashingTF = HashingTF(inputCol='words', 
                      outputCol='rawFeatures',
                      numFeatures=1000)

In [8]:
featurizedData = hashingTF.transform(df_tokens)
idf = IDF(inputCol='rawFeatures', outputCol='features')
idf_model = idf.fit(featurizedData)
df_tfidf = idf_model.transform(featurizedData)

df_tfidf.select('id', 'features').show(truncate=False)

+---+----------------------------------------------------------------------------------------------------------------------------------------------------+
|id |features                                                                                                                                            |
+---+----------------------------------------------------------------------------------------------------------------------------------------------------+
|0  |(1000,[2,270,427,813],[0.4054651081081644,0.4054651081081644,0.4054651081081644,0.4054651081081644])                                                |
|1  |(1000,[169,286,344,472,606,966],[0.4054651081081644,0.4054651081081644,0.4054651081081644,0.4054651081081644,0.4054651081081644,0.4054651081081644])|
+---+----------------------------------------------------------------------------------------------------------------------------------------------------+



## WordToVec

In [9]:
from pyspark.ml.feature import Word2Vec

In [10]:
word2vec = Word2Vec(vectorSize=50, minCount=1, inputCol='words', outputCol='word2vec')
w2v_model = word2vec.fit(df_tokens)
df_w2v = w2v_model.transform(df_tokens)

df_w2v.select('id', 'word2vec').show(truncate=False)

25/09/05 15:48:53 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


+---+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## BERT/Transformer c HuggingFace

In [11]:
import torch
from transformers import AutoTokenizer, AutoModel
from pyspark.sql.functions import pandas_udf
import pandas as pd

In [12]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModel.from_pretrained('bert-base-uncased')

In [13]:
@pandas_udf("array<float>")
def encode_text_udf(texts: pd.Series) -> pd.Series:
    inputes = tokenizer(texts.tolist(), 
                        return_tensors='pt',
                        padding=True,
                        truncation=True)
    with torch.no_grad():
        outputs = model(**inputes)
    embeddings = outputs.last_hidden_state.mean(dim=1)
    
    return pd.Series(embeddings.tolist())

In [14]:
df_hf = df.withColumn('embeddings', encode_text_udf(df.text))

In [15]:
df_hf.show(truncate=True)

+---+--------------------+--------------------+
| id|                text|          embeddings|
+---+--------------------+--------------------+
|  0|Titanic disaster ...|[-0.030532071, -0...|
|  1|Spark ML example ...|[-0.15097445, -0....|
+---+--------------------+--------------------+



In [17]:
spark.stop()